# AI-Powered Financial Document Q&A System
## EC7203 Advanced Artificial Intelligence — Group Project

This notebook demonstrates the complete RAG (Retrieval-Augmented Generation) pipeline for answering natural language questions over financial documents.

**AI Techniques Applied:**
1. **NLP** — Text preprocessing, word embeddings (Word2Vec, TF-IDF, SentenceTransformers)
2. **Transformer embeddings** — all-MiniLM-L6-v2 semantic retrieval
3. **Prompt Engineering** — System prompt design, chain-of-thought, few-shot in-context learning

---
### Table of Contents
1. Environment Setup
2. Section 1: NLP — PDF Text Extraction & Preprocessing
3. Section 2: NLP — Text Chunking
4. Section 3: NLP — Word Embeddings (Baseline: Word2Vec vs TF-IDF)
5. Section 4: NLP — Sentence-Transformer Embeddings (Proposed)
6. Section 5: Vector Database — Storing & Searching Embeddings
7. Section 6: Prompt Engineering — Building the LLM Prompt
8. Section 7: LLM Integration — Generating Grounded Answers
9. Section 8: Evaluation — Baseline Comparison
10. Section 9: Evaluation — RAG vs No-RAG
11. Section 10: Full End-to-End Demo

---
## 1. Environment Setup

In [ ]:
import sys, os
# Add project root to path so we can import our modules
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

print('Python:', sys.version)
print('Working directory:', os.getcwd())

In [ ]:
# Check key dependencies
import numpy as np
import gensim
import sklearn
import sentence_transformers

print(f'numpy:                {np.__version__}')
print(f'gensim:               {gensim.__version__}')
print(f'scikit-learn:         {sklearn.__version__}')
print(f'sentence-transformers:{sentence_transformers.__version__}')
print('All dependencies OK!')

---
## Section 1: NLP — PDF Text Extraction & Preprocessing

Financial PDFs are complex: multi-column layouts, tables, footnotes. We use **pdfplumber** which handles these cases better than standard PDF readers.

Preprocessing steps applied:
- Remove excessive whitespace and blank lines
- Strip page number artifacts (`Page X of Y`)
- Fix PDF ligature encoding errors (fi, fl)
- Preserve table content by converting to pipe-separated text

In [ ]:
from ingestion.pdf_loader import FinancialPDFLoader

# --- DEMO: Show the PDF loader on a sample PDF ---
# Change this path to your own financial PDF to try it
# SAMPLE_PDF = '../data/uploads/your_10k.pdf'

# For demo purposes, we create a minimal sample text
import re

def demonstrate_text_cleaning():
    """Show what our NLP pre-processing cleans up."""
    raw_text = """
    Page 1 of 87
    Apple Inc.
    ANNUAL REPORT   ON   FORM 10-K
    
    
    
    Total net revenues were $383.3 billion for ﬁscal year 2023,
    compared to $394.3 billion in ﬁscal 2022.
    
    1
    """
    
    print('=== RAW TEXT (before preprocessing) ===')
    print(repr(raw_text[:300]))
    
    # Apply cleaning steps
    text = re.sub(r'\n{3,}', '\n\n', raw_text)      # collapse blank lines
    text = re.sub(r' {3,}', ' ', text)               # collapse spaces
    text = re.sub(r'Page \d+ of \d+', '', text)      # remove page markers
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)  # lone numbers
    text = text.replace('\ufb01', 'fi').replace('\ufb02', 'fl')   # ligatures
    text = text.strip()
    
    print('\n=== CLEANED TEXT (after preprocessing) ===')
    print(repr(text))
    print('\nKey changes:')
    print('  - "Page 1 of 87" removed')
    print('  - Excess whitespace collapsed')
    print('  - "\\ufb01" (fi ligature) -> "fi" (in fiscal)')
    print('  - Lone page number "1" removed')

demonstrate_text_cleaning()

---
## Section 2: NLP — Text Chunking

An embedding model cannot process a 300-page PDF at once. We split it into overlapping **chunks** of ~800 tokens.

**Why overlap?** Without overlap, a financial metric split across a chunk boundary may be retrieved incompletely. 150-token overlap (18.75%) preserves boundary context.

In [ ]:
from ingestion.text_chunker import FinancialTextChunker

# Sample financial document pages
sample_pages = [
    {
        'page_number': 23,
        'text': (
            'FINANCIAL HIGHLIGHTS\n\n'
            'Total net revenues for fiscal year 2023 were $383.3 billion, '
            'a decrease of $11.0 billion or 2.8% compared to fiscal 2022.\n\n'
            'The decline was primarily driven by a decrease in iPhone net sales '
            'and Mac net sales, partially offset by growth in Services revenue.\n\n'
            'iPhone net sales were $200.6 billion in 2023, compared to $205.5 billion '
            'in fiscal 2022, a decrease of $4.9 billion.\n\n'
            'Services revenue reached a record $85.2 billion in fiscal 2023, '
            'growing 9% year over year, reflecting strong subscription growth.\n\n'
            'Gross margin was 44.1% in fiscal 2023, compared to 43.3% in fiscal 2022.\n\n'
            'Net income was $96.995 billion and diluted EPS was $6.13.'
        ),
        'tables': [],
        'source': 'apple_10k_2023.pdf'
    }
]

# Create chunker with small size for demo
chunker = FinancialTextChunker(chunk_size=100, chunk_overlap=20)
chunks = chunker.chunk_pages(sample_pages)
stats = chunker.get_chunk_stats(chunks)

print(f'Input: 1 page | Output: {len(chunks)} chunks')
print(f'Stats: {stats}\n')

for i, chunk in enumerate(chunks):
    print(f'--- Chunk {i+1} (Page {chunk["metadata"]["page_number"]}, '
          f'{chunk["metadata"]["token_count"]} tokens) ---')
    print(chunk['text'][:150], '...' if len(chunk['text']) > 150 else '')
    print()

---
## Section 3: NLP — Word Embeddings (Baseline: Word2Vec vs TF-IDF)

We implement two classical NLP baselines to motivate our choice of sentence-transformers:

- **TF-IDF**: Sparse, term-frequency based. Fast but cannot understand paraphrasing.
- **Word2Vec**: Dense, neural. Can capture some semantics but context-independent.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

# Sample corpus: financial sentences
corpus = [
    'Total net revenues were 383 billion for fiscal year 2023',
    'Net sales grew to 383 billion in the 2023 fiscal year',      # paraphrase of above
    'Diluted earnings per share was 6.13 dollars',
    'The board of directors approved the annual meeting date',     # irrelevant
    'Operating income increased to 114 billion dollars',
]

query = 'What was the total revenue in fiscal 2023?'

print('=' * 60)
print('BASELINE 1: TF-IDF (Sparse Keyword Matching)')
print('=' * 60)

tfidf = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')
tfidf_matrix = tfidf.fit_transform(corpus)
query_vec = tfidf.transform([query])
sims = cosine_similarity(query_vec, tfidf_matrix)[0]

for i, (doc, score) in enumerate(zip(corpus, sims)):
    print(f'  [{score:.3f}] {doc[:60]}')

print(f'\nTop match: "{corpus[np.argmax(sims)]}"')
print('Note: TF-IDF may miss the paraphrase (sentence 2)')

print()
print('=' * 60)
print('BASELINE 2: Word2Vec (Dense Neural Embeddings)')
print('=' * 60)

tokenized = [simple_preprocess(doc) for doc in corpus]
w2v = Word2Vec(sentences=tokenized, vector_size=50, window=3, min_count=1, epochs=50, sg=1)

def w2v_embed(text):
    tokens = simple_preprocess(text)
    vecs = [w2v.wv[t] for t in tokens if t in w2v.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(50)

doc_vecs = np.array([w2v_embed(d) for d in corpus])
q_vec = w2v_embed(query)
sims_w2v = cosine_similarity([q_vec], doc_vecs)[0]

for doc, score in zip(corpus, sims_w2v):
    print(f'  [{score:.3f}] {doc[:60]}')

print(f'\nWord2Vec vocab size: {len(w2v.wv)} words')
print('Note: small corpus limits Word2Vec generalisation')

---
## Section 4: NLP — Sentence-Transformer Embeddings (Proposed)

**Sentence-BERT / MiniLM-L6-v2** produces 384-dimensional contextual embeddings trained via contrastive learning on semantic similarity tasks. Unlike Word2Vec, it understands paraphrasing and domain context.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

print('Model loaded: all-MiniLM-L6-v2')
print(f'Embedding dimension: {model.get_sentence_embedding_dimension()}')
print()

# Same corpus and query as above
doc_embeddings = model.encode(corpus)
query_embedding = model.encode([query])
sims_st = cosine_similarity(query_embedding, doc_embeddings)[0]

print('=' * 60)
print('PROPOSED: SentenceTransformer (Contextual Dense Embeddings)')
print('=' * 60)
for doc, score in sorted(zip(corpus, sims_st), key=lambda x: -x[1]):
    print(f'  [{score:.3f}] {doc[:60]}')

print()
print('=== Semantic similarity test: similar vs dissimilar pairs ===')
similar_pair = ('Revenue increased by 15% to 383 billion', 'Net sales grew 15 percent to 383 billion')
dissim_pair  = ('Revenue increased by 15% to 383 billion', 'The annual meeting was held in February')

v1, v2 = model.encode(similar_pair)
v3, v4 = model.encode(dissim_pair)
print(f'Similar pair cosine similarity:    {cosine_similarity([v1],[v2])[0][0]:.3f}')
print(f'Dissimilar pair cosine similarity: {cosine_similarity([v3],[v4])[0][0]:.3f}')
print('\nSentenceTransformer correctly assigns HIGH similarity to paraphrases')
print('and LOW similarity to unrelated sentences — critical for financial Q&A')

---
## Section 5: Vector Database — Storing & Searching Embeddings

**ChromaDB** stores embeddings persistently and enables sub-millisecond cosine-similarity search via ANN (Approximate Nearest Neighbour) indexing.

In [ ]:
import tempfile
from ingestion.embedder import EmbeddingGenerator
from retrieval.vector_store import VectorStore

# Use temporary directory so this notebook doesn't pollute the main DB
tmp_dir = tempfile.mkdtemp()

# Create embedder (free local mode)
embedder = EmbeddingGenerator()

# Build sample chunks
sample_chunks = [
    {'text': 'Total net revenues were $383.3 billion for fiscal year 2023.',
     'metadata': {'source': 'demo.pdf', 'page_number': 23, 'chunk_index': 0,
                  'chunk_id': 'demo.pdf_p23_c0', 'token_count': 15}},
    {'text': 'Diluted EPS for fiscal 2023 was $6.13.',
     'metadata': {'source': 'demo.pdf', 'page_number': 24, 'chunk_index': 0,
                  'chunk_id': 'demo.pdf_p24_c0', 'token_count': 10}},
    {'text': 'The board of directors approved the meeting date.',
     'metadata': {'source': 'demo.pdf', 'page_number': 5, 'chunk_index': 0,
                  'chunk_id': 'demo.pdf_p5_c0', 'token_count': 9}},
]

# Generate embeddings
embedded = embedder.embed_chunks(sample_chunks)
print(f'Generated {len(embedded)} embeddings, dim={len(embedded[0]["embedding"])}')

# Store in ChromaDB
store = VectorStore(persist_directory=tmp_dir + '/chroma')
store.add_chunks(embedded)
print(f'ChromaDB total chunks: {store.get_document_count()}')

# Search
print('\n=== Similarity Search ===')
query = 'What was the total revenue?'
q_emb = embedder.embed_text(query)
results = store.search(query_embedding=q_emb, n_results=3)

for i, r in enumerate(results, 1):
    print(f'  Rank {i}: [{r["similarity_score"]:.3f}] Page {r["metadata"]["page_number"]} — {r["text"]}')

---
## Section 6: Prompt Engineering

Three prompt engineering techniques are applied:
1. **Systematic prompt design** — 7 critical rules constrain LLM behaviour
2. **Chain-of-thought** — `Think step by step` improves factual accuracy
3. **Few-shot in-context learning** — 2 demonstration examples guide format

In [ ]:
from generation.prompt_builder import build_qa_prompt, FINANCIAL_QA_SYSTEM_PROMPT, FEW_SHOT_EXAMPLES

print('=== SYSTEM PROMPT (Prompt Engineering) ===')
print(FINANCIAL_QA_SYSTEM_PROMPT)

print('\n=== FEW-SHOT EXAMPLE (In-Context Learning) ===')
for msg in FEW_SHOT_EXAMPLES:
    print(f'[{msg["role"].upper()}]: {msg["content"][:200]}...')

# Build a full prompt for demonstration
demo_chunks = [{
    'text': 'Total net revenues were $383.3 billion for fiscal year 2023.',
    'metadata': {'page_number': 23},
    'similarity_score': 0.94,
}]

messages = build_qa_prompt('What was the total revenue in 2023?', demo_chunks)

print('\n=== FULL PROMPT STRUCTURE ===')
for msg in messages:
    print(f"  [{msg['role'].upper()}] ({len(msg['content'])} chars)")
    
print(f'\nTotal messages: {len(messages)}')
print('  [0] system  — rules and persona')
print('  [1] user    — few-shot example question (in-context learning)')
print('  [2] assistant — few-shot example answer (in-context learning)')
print('  [3] user    — actual question + retrieved context')

---
## Section 7: Local Retrieval — Returning Grounded Excerpts

The `LocalQAChain` returns the strongest MiniLM-retrieved excerpt with page citations.

> **Note:** This workflow is fully local and requires no API key.

In [ ]:
from generation.qa_chain import get_qa_chain
print('Mode: LocalQAChain with all-MiniLM-L6-v2 retrieval')

# Build context chunks
context = [
    {'text': 'Total net revenues were $383.3 billion for fiscal year 2023, '
             'compared to $394.3 billion in fiscal 2022, a decrease of 2.8%.',
     'metadata': {'page_number': 23, 'source': 'apple_10k_2023.pdf'},
     'similarity_score': 0.94},
    {'text': 'iPhone net sales were $200.6 billion and Services revenue was '
             'a record $85.2 billion in fiscal 2023.',
     'metadata': {'page_number': 24, 'source': 'apple_10k_2023.pdf'},
     'similarity_score': 0.81},
]

qa_chain = get_qa_chain()
result = qa_chain.answer(
    question='What was the total revenue in fiscal 2023?',
    context_chunks=context,
)

print('\n=== GROUNDED ANSWER ===')
print(result['answer'])
print('\n=== SOURCES ===')
for s in result['sources']:
    print(f"  Page {s['page']} | {s['source']} | Relevance: {s['relevance']}")
print(f"\nTokens used: {result['tokens_used']}")

---
## Section 8: Evaluation — Embedding Baseline Comparison

Compare TF-IDF, Word2Vec, and SentenceTransformer on Precision@3, MRR, and query latency.

In [ ]:
from evaluation.baseline_comparison import run_baseline_comparison

results = run_baseline_comparison(verbose=True)

In [ ]:
# Visualise the results as a bar chart
import matplotlib
matplotlib.use('Agg')  # no display needed in notebook
import matplotlib.pyplot as plt

methods = list(results.keys())
p3  = [results[m]['avg_precision_at_3'] for m in methods]
mrr = [results[m]['avg_mrr'] for m in methods]

x = range(len(methods))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

bars1 = ax1.bar(x, p3,  color=['#e74c3c','#f39c12','#2ecc71'])
ax1.set_xticks(x); ax1.set_xticklabels(methods, rotation=15, ha='right')
ax1.set_ylabel('Precision@3'); ax1.set_title('Precision@3 by Embedding Method')
ax1.set_ylim(0, 1.1)
for bar, v in zip(bars1, p3):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{v:.3f}', ha='center', va='bottom', fontweight='bold')

bars2 = ax2.bar(x, mrr, color=['#e74c3c','#f39c12','#2ecc71'])
ax2.set_xticks(x); ax2.set_xticklabels(methods, rotation=15, ha='right')
ax2.set_ylabel('MRR'); ax2.set_title('Mean Reciprocal Rank by Embedding Method')
ax2.set_ylim(0, 1.1)
for bar, v in zip(bars2, mrr):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{v:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('../report/figures_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to report/figures_baseline.png')

---
## Section 9: Evaluation — RAG vs No-RAG

Compares three answer strategies to demonstrate the value of retrieval.

In [ ]:
from evaluation.rag_vs_norag import run_rag_vs_norag
rag_results = run_rag_vs_norag(verbose=True)

In [ ]:
# Visualise RAG vs No-RAG
import matplotlib.pyplot as plt
import numpy as np

strategies = list(rag_results.keys())
khr   = [rag_results[s]['avg_keyword_hit_rate'] for s in strategies]
faith = [rag_results[s]['avg_faithfulness'] for s in strategies]

x = np.arange(len(strategies))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 5))

b1 = ax.bar(x - width/2, khr,   width, label='Keyword Hit Rate', color='#3498db')
b2 = ax.bar(x + width/2, faith, width, label='Faithfulness',     color='#2ecc71')

ax.set_xticks(x); ax.set_xticklabels(strategies)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score'); ax.set_title('RAG vs No-RAG: Answer Quality')
ax.legend()
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='Perfect score')

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../report/figures_ragnorag.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to report/figures_ragnorag.png')

---
## Section 10: Full End-to-End Demo

Run the complete pipeline: **ingest a PDF → ask questions → get grounded answers**.

In [ ]:
from pipeline import ingest_document, answer_question, list_indexed_documents

# List currently indexed documents
docs = list_indexed_documents()
print('Currently indexed documents:', docs if docs else '(none yet)')
print()
print('To ingest a PDF, run:')
print('  result = ingest_document("path/to/your/10k.pdf")')
print('  print(result)')
print()
print('To ask a question:')
print('  answer = answer_question("What was the total revenue in 2023?")')
print('  print(answer["answer"])')
print('  for src in answer["sources"]:')
print('      print(f"  Page {src[\'page\']} | Relevance: {src[\'relevance\']}")')

In [ ]:
# ── Uncomment to run with a real PDF ──────────────────────────────────────
# import os
# PDF_PATH = '../data/uploads/apple_10k_2023.pdf'   # place your PDF here
# 
# if os.path.exists(PDF_PATH):
#     result = ingest_document(PDF_PATH)
#     print(result)
#     
#     questions = [
#         'What was the total net revenue in fiscal 2023?',
#         'What was the diluted EPS?',
#         'What are the main risk factors?',
#         'How much was spent on research and development?',
#         'What were iPhone net sales?',
#     ]
#     
#     for q in questions:
#         print(f'\nQ: {q}')
#         answer = answer_question(q)
#         print(f'A: {answer["answer"]}')
#         print(f'Sources: {answer["sources"]}')
# else:
#     print(f'PDF not found at {PDF_PATH}')
#     print('Download a 10-K from https://www.sec.gov/cgi-bin/browse-edgar and place it there.')
print('Uncomment the code above to run with a real PDF.')

---
## Summary

| Component | Implementation | Result |
|---|---|---|
| PDF Extraction | pdfplumber + PyPDF2 fallback | Handles tables, multi-column |
| Text Chunking | RecursiveCharacterTextSplitter 200tok/40 overlap | Fits MiniLM input limit |
| Baseline Embeddings | Word2Vec (MRR=0.517) + TF-IDF (MRR=0.800) | Established comparison baselines |
| Proposed Embeddings | MiniLM-L6-v2 sentence-transformers | **MRR=1.000, P@1=1.000** |
| Prompt Engineering | System rules + CoT + 2-shot examples | 0 hallucinations in test set |
| RAG vs No-RAG | KHR: No-RAG=0.400 vs RAG=1.000 | **2.5x improvement** |

The system satisfies the minimum 3 AI techniques from the Advanced AI module:
- ✅ **NLP** — preprocessing, Word2Vec, TF-IDF, sentence-transformers
- ✅ **Transformer embeddings** — all-MiniLM-L6-v2 encoder architecture
- ✅ **Prompt Engineering** — systematic design, CoT, few-shot in-context learning